# Setup

In [2]:
import optuna
from optuna.samplers import TPESampler
from optuna.trial import Trial
from optuna.study import create_study
import diff_cont
import lambda_cont
import libs.agent_infra as ai
import os

import json

import datetime
import itertools
import constants as Cs
import concurrent.futures
import numpy as np
from concurrent.futures import ProcessPoolExecutor
from scipy.spatial.distance import pdist

SEEDS = [101, 102, 103]
TEST_EVAL_EPS = 6

/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-17 09:54:05.199192: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-17 09:54:05.243601: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-17 09:54:06.109842: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF

# Lambda

## Fitness

In [ ]:
from scipy.spatial.distance import pdist

def objective(trial:Trial):
    environment = "lunarlander"
    env = Cs.ENIVROMENTS[environment]()
    cross_method = trial.suggest_categorical("crossmethod", ["uniform", "mean"])
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    m = trial.suggest_categorical("mu",[40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0.05, 0.5, step=0.05)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    sigma = trial.suggest_float("sigma", 0.5, 3, step=0.5)
    cross_uni = None
    if cross_method == "uniform":
        cross_uni = 0.5
        #cross_uni = trial.suggest_float("cross", 0.4, 0.6, step=0.1)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.lambda_cont.argumented_function, 
                env=environment,
                container="fitness",
                cross_method=cross_method,
                ng = 15,
                l = l,
                m = m,
                cr = cr,
                mr = mr,
                mutation_sigma = sigma,
                cross_uni = cross_uni if cross_uni is not None else 0.5,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]


    #trial.set_user_attr("scores", fitnesses)
    behaviors = list(map(lambda x:x[1], fitnesses))
    fitnesses = list(map(lambda x:x[0], fitnesses))
    diversity = pdist(np.array(behaviors)).mean()
    assert diversity > 0
    return np.max(fitnesses) + diversity


sampler = TPESampler(n_startup_trials=20)
study = create_study(
    direction="maximize", 
    sampler=sampler, 
    study_name="lambda_fitness_mean", 
    storage="sqlite:///Data/optuna/lunarlander/lambda.db", 
    load_if_exists=True
)
study.optimize(objective, n_trials=110, n_jobs=5)
print(study.best_trials)

## Fit Archiving

In [ ]:


def objective(trial:Trial):
    environment = "lunarlander"
    env = Cs.ENIVROMENTS[environment]()
    cross_method = trial.suggest_categorical("crossmethod", ["uniform", "mean"])
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    m = trial.suggest_categorical("mu",[40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 0.1, step=0.01)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    sigma = trial.suggest_float("sigma", 0.5, 3, step=0.5)
    archiving = trial.suggest_int("archiving_period", 2, 5)
    archive_batch = trial.suggest_int("archive_batch", 1, 5)
    cross_uni = None
    if cross_method == "uniform":
        cross_uni = trial.suggest_float("cross", 0.4, 0.6, step=0.1)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.lambda_cont.argumented_function, 
                env=environment,
                container="fit_archiving",
                cross_method=cross_method,
                ng = 15,
                l = l,
                m = m,
                cr = cr,
                mr = mr,
                mutation_sigma = sigma,
                archiving_period=archiving,
                archive_batch=archive_batch,
                cross_uni = cross_uni if cross_uni is not None else 0.5,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]


    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler()
study = create_study(direction="maximize", study_name="lambda_fit_archiving", sampler=sampler, storage="sqlite:///Data/optuna/lunarlander/fit_archiving/lambda.db", load_if_exists=True)
study.optimize(objective, n_trials=150, n_jobs=5)
print(study.best_trials)

## Novelty Archving

In [ ]:


def objective(trial:Trial):
    environment = "lunarlander"
    env = Cs.ENIVROMENTS[environment]()
    cross_method = trial.suggest_categorical("crossmethod", ["uniform", "mean"])
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    m = trial.suggest_categorical("mu",[40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 0.1, step=0.01)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    sigma = trial.suggest_float("sigma", 0.5, 3, step=0.5)
    archiving = trial.suggest_int("archiving_period", 2, 5)
    archive_batch = trial.suggest_int("archive_batch", 1, 5)
    cross_uni = None
    if cross_method == "uniform":
        cross_uni = trial.suggest_float("cross", 0.4, 0.6, step=0.1)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.lambda_cont.argumented_function, 
                env=environment,
                container="novelty_archiving",
                cross_method=cross_method,
                ng = 15,
                l = l,
                m = m,
                cr = cr,
                mr = mr,
                mutation_sigma = sigma,
                archiving_period=archiving,
                archive_batch=archive_batch,
                cross_uni = cross_uni if cross_uni is not None else 0.5,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]


    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler(n_startup_trials=15)
study = create_study(direction="maximize", study_name="lambda_novelty_archiving", sampler=sampler, storage="sqlite:///Data/optuna/lunarlander/novelty_archiving/lambda.db", load_if_exists=True)
study.optimize(objective, n_trials=150, n_jobs=5)
print(study.best_trials)

In [ ]:
studies = optuna.get_all_study_summaries(
    storage="sqlite:///Data/optuna/lunarlander/fit_archiving/lambda_alt.db"
)
print(studies)
for s in studies:
    print(s.study_name)
    print(s.n_trials)

## Novelty

In [ ]:
def objective(trial:Trial):
    environment = "lunarlander"
    env = Cs.ENIVROMENTS[environment]()
    cross_method = trial.suggest_categorical("crossmethod", ["uniform", "mean"])
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    m = trial.suggest_categorical("mu",[40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 0.5, step=0.01)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    sigma = trial.suggest_float("sigma", 0.5, 3, step=0.5)

    cross_uni = None
    if cross_method == "uniform":
        cross_uni = trial.suggest_float("cross", 0.1, 0.9, step=0.1)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.lambda_cont.argumented_function, env=environment,
                container="novelty",
                cross_method=cross_method,
                ng = 15,
                l = l,
                m = m,
                cr = cr,
                mr = mr,
                mutation_sigma = sigma,
                cross_uni = cross_uni if cross_uni is not None else 0.5,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]


    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler()
study = create_study(direction="maximize", study_name="lambda_novelty", sampler=sampler, storage="sqlite:///Data/optuna/lunarlander/novelty/lambda.db", load_if_exists=True)
study.optimize(objective, n_trials=150, n_jobs=5)
print(study.best_trials)

In [ ]:
study.best_trials

## Add Novelty

In [ ]:
def objective(trial:Trial):
    env = Cs.ENIVROMENTS["lunarlander"]()
    cross_method = trial.suggest_categorical("crossmethod", ["uniform", "mean"])
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    m = trial.suggest_categorical("mu",[40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 0.5, step=0.01)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    sigma = trial.suggest_float("sigma", 0.5, 3, step=0.5)
    fit_w = trial.suggest_float("start_fit_w", 0.2, 1, step=0.1)
    decay = trial.suggest_float("decay", 0.5, 5, step=0.5)

    cross_uni = None
    if cross_method == "uniform":
        cross_uni = trial.suggest_float("cross", 0.1, 0.9, step=0.1)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.lambda_cont.argumented_function, 
                env="lunarlander",
                container="add_novelty",
                cross_method=cross_method,
                ng = 15,
                l = l,
                m = m,
                cr = cr,
                mr = mr,
                mutation_sigma = sigma,
                cross_uni = cross_uni if cross_uni is not None else 0.5,
                fitness_weight=fit_w,
                decay = decay,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]
    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler(n_startup_trials=25)
study = create_study(direction="maximize", sampler=sampler, study_name="lambda_add_novelty_exp_fixed_inspired", storage="sqlite:///Data/optuna/lunarlander/add_novelty/lambda.db", load_if_exists=True)
study.optimize(objective, n_trials=180, n_jobs=5)
print(study.best_trials)

[I 2026-06-16 06:53:56,311] Using an existing study with name 'lambda_add_novelty_exp_fixed' instead of creating a new one.


   	      	                            fitness                            	                                novelty                                 
   	      	---------------------------------------------------------------	------------------------------------------------------------------------
gen	nevals	avg    	gen	max     	min     	nevals	std    	avg     	gen	max     	min      	nevals	std     
0  	40    	-389.17	0  	-125.803	-602.368	40    	136.448	0.382865	0  	0.800395	0.0215221	40    	0.214543
   	      	                            fitness                            	                                novelty                                 
   	      	---------------------------------------------------------------	------------------------------------------------------------------------
gen	nevals	avg     	gen	max    	min     	nevals	std    	avg     	gen	max     	min       	nevals	std     
0  	40    	-444.135	0  	-100.56	-731.976	40    	178.002	0.381899	0  	0.800405	0.00291978	40    

[I 2026-06-16 07:27:54,568] Trial 9 finished with value: -98.52126239807221 and parameters: {'crossmethod': 'uniform', 'lambda': 50, 'mu': 40, 'mutation_rate': 0.29, 'cross_rate': 0.4, 'sigma': 2.0, 'start_fit_w': 0.2, 'decay': 4.0, 'cross': 0.30000000000000004}. Best is trial 9 with value: -98.52126239807221.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessTrue' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/promet

15 	50    	-70.6716	15 	-58.7203	-99.1076	50    	5.90791	0.916723	15 	0.942541	0.847612   	50    	0.0170923
14 	47    	-72.0189	14 	-67.3705	-108.33 	47    	10.9382	0.90955 	14 	0.934731	0.853422 	47    	0.0240409
12 	60    	-369.598	12 	-61.8472	-555.793	60    	210.452	0.838485	12 	0.96395 	0.31148   	60    	0.109644 
14 	42    	-95.8437	14 	-95.8437	-95.8437	42    	0      	0.968546	14 	0.975741	0.89075   	42    	0.0194755
13 	60    	-57.02  	13 	-7.12086	-100.339	60    	20.7169	0.949032	13 	0.994814	0.808189  	60    	0.0546547
11 	60    	2.20845 	11 	56.1409 	-88.7409	60    	32.8362	0.920826	11 	0.991609	0.793777   	60    	0.0683189
14 	60    	-116.796	14 	-69.816 	-125.803	60    	17.8375	0.992024	14 	1       	0.984785 	60    	0.00544695
   	      	                            fitness                            	                                novelty                                 
   	      	---------------------------------------------------------------	---------------------------

[I 2026-06-16 07:33:45,282] Trial 6 finished with value: -78.4519276512324 and parameters: {'crossmethod': 'uniform', 'lambda': 50, 'mu': 50, 'mutation_rate': 0.33, 'cross_rate': 0.5, 'sigma': 0.5, 'start_fit_w': 0.6000000000000001, 'decay': 3.0, 'cross': 0.2}. Best is trial 6 with value: -78.4519276512324.


...

[I 2026-06-17 09:40:06,825] Trial 179 finished with value: 95.31911731107083 and parameters: {'crossmethod': 'uniform', 'lambda': 70, 'mu': 70, 'mutation_rate': 0.05, 'cross_rate': 0.9000000000000001, 'sigma': 0.5, 'start_fit_w': 0.2, 'decay': 4.5, 'cross': 0.9}. Best is trial 140 with value: 120.23280584297873.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/p

5  	67    	-87.8413	5  	10.5508	-186.702	67    	37.9944	0.859041	5  	0.955456	0.674778  	67    	0.0600063
5  	60    	-69.9157	5  	28.306  	-287.624	60    	48.383 	0.82644 	5  	0.94516 	0.537004 	60    	0.0567723
15 	66    	91.5555 	15 	96.869  	78.1188 	66    	4.09352	0.989898	15 	0.997985	0.927717 	66    	0.00974228
6  	60    	-87.1422	6  	-1.4693 	-125.803	60    	28.6726	0.893953	6  	0.966971	0.752102   	60    	0.0396507
   	      	                            fitness                            	                                novelty                                 
   	      	---------------------------------------------------------------	------------------------------------------------------------------------
gen	nevals	avg     	gen	max    	min     	nevals	std    	avg     	gen	max     	min       	nevals	std     
0  	70    	-410.148	0  	2.20057	-731.976	70    	187.021	0.365898	0  	0.801468	0.00307456	70    	0.197449
   	      	                                fitness                 

[I 2026-06-17 09:50:40,778] Trial 180 finished with value: 95.31911731107083 and parameters: {'crossmethod': 'uniform', 'lambda': 70, 'mu': 70, 'mutation_rate': 0.05, 'cross_rate': 0.9000000000000001, 'sigma': 0.5, 'start_fit_w': 0.2, 'decay': 4.5, 'cross': 0.9}. Best is trial 140 with value: 120.23280584297873.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/p

9  	63    	-23.9452	9  	5.10133 	-48.5539	63    	16.699 	0.940254	9  	0.990596	0.868852   	63    	0.0373751
8  	62    	3.3794  	8  	68.595 	-106.253	62    	50.8007	0.941401	8  	0.982451	0.874859  	62    	0.0230278
8  	64    	-32.9636	8  	34.7491 	-217.615	64    	57.4263	0.899369	8  	0.981962	0.650253 	64    	0.0693006
15 	68    	11.377  	15 	21.8645 	-82.4184	68    	19.9806	0.984892	15 	0.9978  	0.777605   	68    	0.031755 
13 	66    	77.6427 	13 	94.0909 	6.62093 	66    	24.0658	0.984043	13 	0.994995	0.905723 	66    	0.0141143
   	      	                            fitness                            	                                novelty                                 
   	      	---------------------------------------------------------------	------------------------------------------------------------------------
gen	nevals	avg     	gen	max    	min     	nevals	std    	avg     	gen	max     	min       	nevals	std     
0  	70    	-410.148	0  	2.20057	-731.976	70    	187.021	0.365898	

Process ForkProcess-531:
Process ForkProcess-328:
Process ForkProcess-329:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._targ

## Su bNovelty

In [3]:

def objective(trial:Trial):
    env = Cs.ENIVROMENTS["lunarlander"]()
    cross_method = trial.suggest_categorical("crossmethod", ["uniform", "mean"])
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    m = trial.suggest_categorical("mu",[40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 0.5, step=0.01)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    sigma = trial.suggest_float("sigma", 0.5, 3, step=0.5)
    fit_w = trial.suggest_float("start_fit_w", 0.2, 1, step=0.1)
    decay = trial.suggest_float("decay", 0.5, 5, step=0.5)

    cross_uni = None
    if cross_method == "uniform":
        cross_uni = trial.suggest_float("cross", 0.1, 0.9, step=0.1)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.lambda_cont.argumented_function, 
                env="lunarlander",
                container="sub_novelty",
                cross_method=cross_method,
                ng = 15,
                l = l,
                m = m,
                cr = cr,
                mr = mr,
                mutation_sigma = sigma,
                cross_uni = cross_uni if cross_uni is not None else 0.5,
                fitness_weight=fit_w,
                decay = decay,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]
    #trial.set_user_attr("scores", fitnesses)
    #trial.set_user_attr("scores", fitnesses)
    behaviors = list(map(lambda x:x[1], fitnesses))
    fitnesses = list(map(lambda x:x[0], fitnesses))
    diversity = pdist(np.array(behaviors)).mean()
    assert diversity > 0
    return np.max(fitnesses) + diversity


sampler = TPESampler(n_startup_trials=15)
study = create_study(direction="maximize", sampler=sampler, study_name="lambda_sub_novelty_exp_fixed", storage="sqlite:///Data/optuna/lunarlander/sub_novelty/lambda.db", load_if_exists=True)
study.optimize(objective, n_trials=170, n_jobs=5)
print(study.best_trials)

[I 2026-06-17 09:54:08,046] A new study created in RDB with name: lambda_sub_novelty_exp_fixed


   	      	                            fitness                            	                                novelty                                 
   	      	---------------------------------------------------------------	------------------------------------------------------------------------
gen	nevals	avg    	gen	max     	min     	nevals	std    	avg     	gen	max     	min      	nevals	std     
0  	40    	-389.17	0  	-125.803	-602.368	40    	136.448	0.189367	0  	0.819546	0.0490978	40    	0.152923
   	      	                            fitness                            	                                novelty                                 
   	      	---------------------------------------------------------------	------------------------------------------------------------------------
gen	nevals	avg     	gen	max    	min     	nevals	std    	avg     	gen	max     	min       	nevals	std     
0  	40    	-444.135	0  	-100.56	-731.976	40    	178.002	0.159993	0  	0.822224	0.00994886	40    

[I 2026-06-17 10:30:25,148] Trial 4 finished with value: -110.22295088968345 and parameters: {'crossmethod': 'uniform', 'lambda': 40, 'mu': 40, 'mutation_rate': 0.14, 'cross_rate': 0.9000000000000001, 'sigma': 0.5, 'start_fit_w': 0.2, 'decay': 0.5, 'cross': 0.7000000000000001}. Best is trial 4 with value: -110.22295088968345.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessTrue' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv

   	      	                                fitness                                	                            novelty                             
   	      	-----------------------------------------------------------------------	----------------------------------------------------------------
gen	nevals	avg     	gen	max     	min     	nevals	std    	avg    	gen	max     	min        	nevals	std     
0  	50    	-376.934	0  	-125.803	-616.413	50    	145.317	0.45042	0  	0.900136	0.000128056	50    	0.259223
   	      	                            fitness                            	                                novelty                                 
   	      	---------------------------------------------------------------	------------------------------------------------------------------------
gen	nevals	avg     	gen	max    	min     	nevals	std    	avg     	gen	max     	min      	nevals	std     
0  	50    	-442.715	0  	-100.56	-731.976	50    	174.705	0.420969	0  	0.900173	0.0015064	50  

[I 2026-06-17 10:45:32,755] Trial 3 finished with value: -136.8647561968764 and parameters: {'crossmethod': 'mean', 'lambda': 50, 'mu': 70, 'mutation_rate': 0.4, 'cross_rate': 0.3, 'sigma': 0.5, 'start_fit_w': 0.5, 'decay': 2.0}. Best is trial 4 with value: -110.22295088968345.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packa

...

1  	26    	-303.312	1  	-112.238	-682.89	26    	164.479	0.565993	1  	0.834584	0.121144	26    	0.204872
2  	22    	-174.727	2  	-117.123	-417.859	22    	58.8084	0.787266	2  	0.900148	0.381864   	22    	0.116106
2  	17    	-120.676	2  	2.20057	-399.55 	17    	80.2292	0.734191	2  	0.900734	0.429467  	17    	0.108354


[I 2026-06-18 08:58:52,904] Trial 163 finished with value: 26.552243184402613 and parameters: {'crossmethod': 'mean', 'lambda': 70, 'mu': 60, 'mutation_rate': 0.42, 'cross_rate': 0.3, 'sigma': 3.0, 'start_fit_w': 1.0, 'decay': 2.5}. Best is trial 114 with value: 65.49090275974416.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-pa

3  	21    	-153.347	3  	-117.123	-288    	21    	39.5315	0.848153	3  	0.900148	0.602712   	21    	0.0722769
2  	27    	-201.965	2  	-97.1772	-682.89	27    	118.745	0.689089	2  	0.834584	0.168127	27    	0.132846
3  	25    	-64.5937	3  	2.20057	-184.613	25    	61.2457	0.819936	3  	0.900734	0.674078  	25    	0.0729569
3  	15    	-151.389	3  	-97.1772	-366.181	15    	44.0825	0.773829	3  	0.834584	0.556088	15    	0.0487952
4  	24    	-132.7  	4  	-90.416 	-169.884	24    	12.5915	0.879604	4  	0.900148	0.385473   	24    	0.0629101
5  	13    	-126.74 	5  	-125.803	-144.663	13    	3.4664 	0.898456	5  	0.900148	0.867526   	13    	0.00621339
   	      	                                fitness                                	                                novelty                                 
   	      	-----------------------------------------------------------------------	------------------------------------------------------------------------
gen	nevals	avg     	gen	max     	min     	nevals	

[I 2026-06-18 09:12:51,914] Trial 165 finished with value: 27.596550883945973 and parameters: {'crossmethod': 'mean', 'lambda': 70, 'mu': 60, 'mutation_rate': 0.43, 'cross_rate': 0.3, 'sigma': 3.0, 'start_fit_w': 0.9000000000000001, 'decay': 3.0}. Best is trial 114 with value: 65.49090275974416.
[I 2026-06-18 09:15:18,407] Trial 166 finished with value: 27.640347095491784 and parameters: {'crossmethod': 'mean', 'lambda': 70, 'mu': 60, 'mutation_rate': 0.44, 'cross_rate': 0.3, 'sigma': 3.0, 'start_fit_w': 0.9000000000000001, 'decay': 3.0}. Best is trial 114 with value: 65.49090275974416.
[I 2026-06-18 09:18:44,851] Trial 167 finished with value: 28.03882537672636 and parameters: {'crossmethod': 'mean', 'lambda': 70, 'mu': 60, 'mutation_rate': 0.07, 'cross_rate': 0.3, 'sigma': 3.0, 'start_fit_w': 0.9000000000000001, 'decay': 3.0}. Best is trial 114 with value: 65.49090275974416.
[I 2026-06-18 09:19:19,086] Trial 168 finished with value: 27.91741834281711 and parameters: {'crossmethod': '

[FrozenTrial(number=114, state=<TrialState.COMPLETE: 1>, values=[65.49090275974416], datetime_start=datetime.datetime(2026, 6, 18, 2, 11, 23, 684562), datetime_complete=datetime.datetime(2026, 6, 18, 2, 52, 37, 711966), params={'crossmethod': 'mean', 'lambda': 70, 'mu': 40, 'mutation_rate': 0.02, 'cross_rate': 0.3, 'sigma': 3.0, 'start_fit_w': 0.8, 'decay': 2.5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'crossmethod': CategoricalDistribution(choices=('uniform', 'mean')), 'lambda': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mu': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mutation_rate': FloatDistribution(high=0.5, log=False, low=0.0, step=0.01), 'cross_rate': FloatDistribution(high=1.0, log=False, low=0.3, step=0.1), 'sigma': FloatDistribution(high=3.0, log=False, low=0.5, step=0.5), 'start_fit_w': FloatDistribution(high=1.0, log=False, low=0.2, step=0.1), 'decay': FloatDistribution(high=5.0, log=False, low=0.5, step=0.5)}, trial_id=290, v

# Diff

## Fitness

In [ ]:


def objective(trial:Trial):
    environment = "lunarlander"
    env = Cs.ENIVROMENTS[environment]()
    l = trial.suggest_categorical("pop",[40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 0.1, step=0.01)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, 
                env=environment,
                container="fitness",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]


    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler()
diff_fitness_study = create_study(direction="maximize", sampler=sampler, storage="sqlite:///Data/optuna/lunarlander/diff.db", load_if_exists=True)
diff_fitness_study.optimize(objective, n_trials=120, n_jobs=5)
print(diff_fitness_study.best_trials)

## Fit_Archiving

In [ ]:
def objective(trial:Trial):
    environment = "lunarlander"
    env = Cs.ENIVROMENTS[environment]()
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 1, step=0.1)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    archiving = trial.suggest_int("archiving_period", 2, 5)
    archive_batch = trial.suggest_int("archive_batch", 1, 5)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, env=environment,
                container="fit_archiving",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                archiving_period=archiving,
                archive_batch=archive_batch,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]


    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler()
diff_fita_study = create_study(direction="maximize", sampler=sampler, storage="sqlite:///Data/optuna/lunarlander/fit_archiving/diff.db", load_if_exists=True)
diff_fita_study.optimize(objective, n_trials=150, n_jobs=5)
print(diff_fita_study.best_trials)

In [ ]:
#new_study = optuna.load_study(study_name="fitarchiving",storage="sqlite:///Data/optuna/lunarlander/fit_archiving/diff.db")
store = "sqlite:///Data/optuna/lunarlander/lambda.db"
studies = optuna.get_all_study_summaries(
    storage=store
)
for s in studies:
    print(s.study_name)
    print(s.n_trials)

new_study = optuna.load_study(study_name="no-name-e88fa580-a490-41e6-bbb1-5aa576e4e219",storage=store)

new_study.best_trials

## Novelty

In [ ]:
def objective(trial:Trial):
    environment = "lunarlander"
    env = Cs.ENIVROMENTS[environment]()
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 0.5, step=0.01)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, env=environment,
                container="novelty",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]


    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler()
diff_novelty_study = create_study(direction="maximize", sampler=sampler, study_name="diff_novelty", storage="sqlite:///Data/optuna/lunarlander/novelty/diff.db", load_if_exists=True)
diff_novelty_study.optimize(objective, n_trials=150, n_jobs=5)
print(diff_novelty_study.best_trials)

In [ ]:
#new_study = optuna.load_study(study_name="fitarchiving",storage="sqlite:///Data/optuna/lunarlander/fit_archiving/diff.db")
studies = optuna.get_all_study_summaries(
    storage="sqlite:///Data/optuna/lunarlander/fit_archiving/diff.db"
)
print(studies)
for s in studies:
    print(s.study_name)
    print(s.n_trials)

new_study = optuna.load_study(study_name="no-name-a6a24196-d355-4f39-af93-68a295e37c48",storage="sqlite:///Data/optuna/lunarlander/fit_archiving/diff.db")

new_study.best_trials